<h1><font color="#113D68" size=5>Deep Learning para Procesamiento del Lenguaje Natural</font></h1>



<h1><font color="#113D68" size=6>3. Preparar datos con Keras</font></h1>



---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>

* [0. Contexto](#section0)
* [1. Dividir palabras con `text_to_word_sequence`](#section1)
* [2. Codificación con  `one_hot`](#section2)
* [3. Codificación hash con `hashing_trick`](#section3)
* [4. API `Tokenizer`](#section4)

---
<a id="section0"></a>
# <font color="#004D7F" size=6> 0. Contexto</font>

Keras proporciona algunas herramientas para convertir datos de formato texto a numérico para preparar un corpus que pueda ser ejecutado por los modelos. En este tutorial trabajaremos:
- Los métodos para procesar datos de texto.
- La API `Tokenizer` que codifica documentos, y realiza el proceso de validación y prueba.
- Los 4 esquemas de codificación de documentos diferentes que ofrece `Tokenizer`.

<a id="section1"></a>
# <font color="#004D7F" size=6>1. Dividir palabras con `text_to_word_sequence`</font>

Un buen primer paso cuando se trabaja con texto es dividirlo en palabras. Las palabras se llaman __tokens__ y el proceso de dividir el texto en tokens se denomina __tokenización__. Keras proporciona la función `text_to_word_sequence()` que divide el texto en una lista de palabras, realizando tres cosas:
1. Dividir palabras por espacios en blanco.
2. Filtrar la puntuación.
3. Convertir texto a minúsculas (`lower=True`).

Puede cambiar cualquiera de estos valores predeterminados pasando argumentos a la función.

In [7]:
import keras
print(keras.__version__)

3.14.0


<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre la clase [`text_to_word_sequence`](https://faroit.com/keras-docs/2.0.5/preprocessing/text/#text_to_word_sequence)

In [9]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

# 1. Definición de un texto de prueba (Con mayúsculas, puntuación y guiones)
texto_sucio = "¡Gregor Samsa se despertó una mañana! Tenía un sueño tranquilo... ¿o no?"

# 2. Aplicación de la función con sus parámetros por defecto
# Por defecto, elimina los caracteres de string.punctuation más algunos específicos como ¡ y ¿
lista_palabras = text_to_word_sequence(texto_sucio)

# ==============================================================================
# VISUALIZACIÓN COMPLETA DEL ANTES Y EL DESPUÉS
# ==============================================================================
print("=== TEXTO ORIGINAL DE ENTRADA ===")
print(f"\"{texto_sucio}\"")
print(f"Tipo de dato: {type(texto_sucio)}")
print("\n" + "="*60 + "\n")

print("=== RESULTADO DE text_to_word_sequence ===")
print(lista_palabras)
print(f"Tipo de dato: {type(lista_palabras)}")
print(f"Total de tokens extraídos: {len(lista_palabras)}")

=== TEXTO ORIGINAL DE ENTRADA ===
"¡Gregor Samsa se despertó una mañana! Tenía un sueño tranquilo... ¿o no?"
Tipo de dato: <class 'str'>


=== RESULTADO DE text_to_word_sequence ===
['¡gregor', 'samsa', 'se', 'despertó', 'una', 'mañana', 'tenía', 'un', 'sueño', 'tranquilo', '¿o', 'no']
Tipo de dato: <class 'list'>
Total de tokens extraídos: 12


<a id="section2"></a>
# <font color="#004D7F" size=6>2. Codificación con `one_hot`</font>

Keras proporciona la función `one_hot()` que se puede usar para tokenizar y codificar un documento de texto en un solo paso. 
- El nombre sugiere que creará un one-hot encoding, lo cual no es el caso. 
- La función es un wrapper para la función `hashing_trick()`. 
- La función devuelve una versión codificada en enteros del documento. 
- El uso de una función hash significa que puede haber colisiones y no a todas las palabras se les asignarán valores enteros únicos. 
- `one_hot()` hará que el texto esté en minúsculas, filtrará la puntuación y dividirá las palabras en función de los espacios en blanco.
- Además del texto, se debe especificar el tamaño del vocabulario (palabras totales).
- El tamaño del vocabulario define el espacio hash desde el cual se codifican las palabras.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre la clase [`one_hot`](https://faroit.com/keras-docs/2.0.5/preprocessing/text/#one_hot)

Primero vamos a verificar el tamaño del vocabulario.

In [14]:
import string
from tensorflow.keras.preprocessing.text import one_hot
# Corrección técnica: Es obligatorio importar text_to_word_sequence para poder usarla en la celda
from tensorflow.keras.preprocessing.text import text_to_word_sequence

# Texto de prueba
texto = "Gregor Samsa despertó una mañana de un sueño tranquilo"

# 1. Extracción de fichas limpias (Tokens)
# text_to_word_sequence limpia la puntuación y pasa a minúsculas. 
# Al envolverlo en set(), eliminamos duplicados obteniendo la colección de términos únicos.
words = set(text_to_word_sequence(texto))

# 2. Corrección del tamaño del vocabulario (vocab_size)
# ERROR ANTERIOR: vocab_size = words (Pasaba un objeto 'set', provocando un TypeError).
# SOLUCIÓN: Usamos len(words) para obtener el número entero de palabras únicas (en este caso, 9).
# Nota de diseño: Tradicionalmente se le añade un margen (ej. multiplicarlo por 1.5 o un entero fijo)
# para reducir la probabilidad de colisiones de hash internas dentro de la función one_hot.
vocab_size = len(words)

**Factor de Carga (Load Factor)**, y se aplica específicamente porque la función one_hot de Keras funciona mediante un algoritmo de Tabla de Hash.

Cuando usas <code> one_hot </code> de Keras, el modelo no crea un diccionario ordenado para recordar qué palabra corresponde a qué número; en su lugar, toma cada palabra, calcula un código matemático (su hash) y lo divide entre el tamaño del vocabulario (vocab_size) para obtener un residuo (el índice de la columna).

**1. El problema del "Efecto Cumpleaños" y las Colisiones**

Si tienes exactamente 9 palabras únicas y configuras tu vocab_size = 9, la probabilidad matemática de que dos palabras totalmente distintas calculen por casualidad el mismo residuo y caigan en el mismo casillero es altísima. Esto se conoce como colisión de hash.
Si ocurre una colisión, tu red neuronal recibirá el mismo ID para dos conceptos diferentes (por ejemplo, que tanto 'sueño' como 'mañana' devuelvan el ID 4), lo que arruinará la capacidad del modelo para diferenciar los términos.

In [16]:
# 3. Aplicación del algoritmo One-Hot basado en Hashing de Keras
# Convierte cada palabra del string original en un ID entero indexado de forma determinista.
# El rango de los IDs resultantes estará acotado estrictamente entre 1 y el valor de 'vocab_size'.
#
# Nota de diseño importante: Multiplicar por 1.3 aplica formalmente la regla del "tercer tercio" 
# de espacio adicional (un incremento del 30%) para diluir la densidad de la tabla de hash.
#
# Alerta de Tipo de Dato (Data Type): La función round() en Python devuelve un valor de tipo float 
# cuando se usa en ciertas operaciones complejas o versiones previas, pero 'one_hot' exige estrictamente 
# un número entero (int). Para blindar tu código contra errores de tipo (TypeError) en Colab, 
# la mejor práctica de ingeniería de datos es envolver la operación en la función nativa int().
resultado_ids = one_hot(texto, n=int(round(vocab_size * 1.3)))

# ==============================================================================
# MONITOREO VISUAL EN COLAB
# ==============================================================================
# Este bloque imprime las variables de control en la consola de tu cuaderno de Jupyter
# para que puedas auditar y contrastar la dimensión original versus la dimensión inflada con hash.
print("=== ENFOQUE 1: REPRESENTACIÓN DE IDS DE UNA COMPRESIÓN HASH ===")
print(f"Texto original: {texto}")
print(f"Vocab Size base (Palabras únicas reales): {vocab_size}")
# Mostramos el valor real que recibió el parámetro 'n' para verificar el crecimiento de la tabla
print(f"Vocab Size escalado (Capacidad total de la tabla con +30%): {int(round(vocab_size * 1.3))}")
print(f"Lista de IDs resultante de Keras (Secuencia mapeada): {resultado_ids}")

=== ENFOQUE 1: REPRESENTACIÓN DE IDS DE UNA COMPRESIÓN HASH ===
Texto original: Gregor Samsa despertó una mañana de un sueño tranquilo
Vocab Size base (Palabras únicas reales): 9
Vocab Size escalado (Capacidad total de la tabla con +30%): 12
Lista de IDs resultante de Keras (Secuencia mapeada): [1, 8, 7, 11, 8, 5, 6, 7, 7]


Podemos juntar esto con la función `one_hot()` y codificar las palabras en el documento. 
- El tamaño del vocabulario se incrementa en un tercio para minimizar las colisiones al mezclar palabras.
- El documento codificado se imprime como una matriz de palabras codificadas con números enteros.

Nota: Dada la naturaleza estocástica de las redes neuronales, sus resultados específicos pueden variar. Considere ejecutar el ejemplo varias veces.

<a id="section3"></a>
# <font color="#004D7F" size=6>3. Codificación hash con `hashing_trick`</font>

- Keras proporciona la función `hashing_trick()` que tokeniza y luego codifica el documento con enteros, al igual que la función `one_hot()`. 
- Proporciona más flexibilidad, lo que le permite especificar la función hash como `hash` (predeterminada) u otras funciones, como la función `md5`. 


In [21]:
import hashlib
import string
from tensorflow.keras.preprocessing.text import hashing_trick
from tensorflow.keras.preprocessing.text import text_to_word_sequence

# ==============================================================================
# 1. DEFINICIÓN DE LA FUNCIÓN EN JUGADA (MD5)
# ==============================================================================
def funcion_hash_md5(palabra):
    """
    Toma un string, calcula su hash MD5, lo convierte a su representación 
    hexadecimal y lo transforma en un entero de base 16 (Big Int).
    Esto garantiza un comportamiento 100% determinista.
    """
    hash_objeto = hashlib.md5(palabra.encode('utf-8'))
    return int(hash_objeto.hexdigest(), 16)

# ==============================================================================
# 2. CONFIGURACIÓN DEL CORPUS Y VARIABLES DE CONTROL
# ==============================================================================
texto = "Gregor Samsa despertó una mañana de un sueño tranquilo"

# Análisis de dimensiones base
words = set(text_to_word_sequence(texto))
palabras_unicas = len(words) # 9 palabras únicas

# Aplicamos la regla del tercio (+30% de espacio) para el factor de carga
vocab_size_escalado = int(round(palabras_unicas * 1.3)) # Resultado: 12

# ==============================================================================
# 3. EJECUCIÓN DE HASHING_TRICK CON MD5
# ==============================================================================
resultado_trick_ids = hashing_trick(
    texto, 
    n=vocab_size_escalado, 
    hash_function=funcion_hash_md5, # Pasamos la función ejecutable, NO un string
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
    lower=True
)

# ==============================================================================
# MONITOREO VISUAL EN COLAB / ANACONDA
# ==============================================================================
print("=== HASHING_TRICK CON FUNCIÓN DE ENTRADA MD5 ===")
print(f"Texto original:             {texto}")
print(f"Capacidad fijada (n):       {vocab_size_escalado}")
print(f"Rango de IDs posibles:      [1 a {vocab_size_escalado - 1}]")
print("-" * 60)
print(f"Secuencia de IDs (MD5):     {resultado_trick_ids}")

=== HASHING_TRICK CON FUNCIÓN DE ENTRADA MD5 ===
Texto original:             Gregor Samsa despertó una mañana de un sueño tranquilo
Capacidad fijada (n):       12
Rango de IDs posibles:      [1 a 11]
------------------------------------------------------------
Secuencia de IDs (MD5):     [3, 5, 1, 1, 6, 10, 9, 2, 8]


<a id="section4"></a>
# <font color="#004D7F" size=6>4. API de `Tokenizador`</font>

Keras proporciona la API `Tokenizer` para preparar texto que se puede ajustar y reutilizar para preparar varios documentos de texto.

`Tokenizer` debe construirse y luego caber en documentos de texto sin formato o documentos de texto codificados con enteros.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre la clase [`Tokenizer`](https://faroit.com/keras-docs/2.0.5/preprocessing/text/#tokenizer)

In [40]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer

# 1. Definición del corpus (Mismo ejemplo en español)
corpus = [
    'Gregor Samsa despertó una mañana de un sueño tranquilo.',
    'El sueño de Gregor no era un sueño normal.',
    'Una mañana tranquila y una tarde tranquila.'
]

# 2. Instanciación del Tokenizer
# - num_words: El tamaño máximo de tu vocabulario (similar al vocab_size). Solo se quedará con el Top N.
# - oov_token: El token especial que usará Keras cuando encuentre una palabra desconocida en producción.
# - lower=True: Convierte de forma nativa todo el texto a minúsculas.
# - filters: Filtra automáticamente los signos de puntuación por defecto.
tokenizer = Tokenizer(
    #num_words=20, 
    oov_token="[UNK]", 
    lower=True
)

# 3. Construcción del vocabulario interno (Fase de Ajuste / FIT)
# El Tokenizer lee todo el corpus, cuenta frecuencias y genera los índices deterministas.
tokenizer.fit_on_texts(corpus)

`Tokenizer` proporciona 4 atributos:
- __`word_counts`__: diccionario de palabras y sus recuentos de ocurrencia.
- __`document_count`__: número de documentos evaluados.
- __`word_index`__: diccionario de palabras y sus índices únicos.
- __`word_docs`__: diccionario de palabras y el número de documentos que aparecen.


In [42]:
# 4. Extracción de los diccionarios internos aprendidos por Keras
# - word_index: Muestra el mapeo de palabra a su ID numérico asignado.
# - word_counts: Muestra cuántas veces se repitió cada palabra en todo el corpus.
diccionario_vocabulario = tokenizer.word_index
conteo_frecuencias = tokenizer.word_counts

print(f"""
corpus : 
{corpus}

word_counts (diccionario de palabras y sus recuentos de ocurrencia.): 
{tokenizer.word_counts}

documen_count ( número de documentos evaluados) : {tokenizer.document_count} 

word_index (diccionario de palabras y sus índices únicos) : {tokenizer.word_index}

word_docs (diccionario de palabras y el número de documentos que aparecen.): {tokenizer.word_docs}
""")



corpus : 
['Gregor Samsa despertó una mañana de un sueño tranquilo.', 'El sueño de Gregor no era un sueño normal.', 'Una mañana tranquila y una tarde tranquila.']

word_counts (diccionario de palabras y sus recuentos de ocurrencia.): 
OrderedDict([('gregor', 2), ('samsa', 1), ('despertó', 1), ('una', 3), ('mañana', 2), ('de', 2), ('un', 2), ('sueño', 3), ('tranquilo', 1), ('el', 1), ('no', 1), ('era', 1), ('normal', 1), ('tranquila', 2), ('y', 1), ('tarde', 1)])

documen_count ( número de documentos evaluados) : 3 

word_index (diccionario de palabras y sus índices únicos) : {'[UNK]': 1, 'una': 2, 'sueño': 3, 'gregor': 4, 'mañana': 5, 'de': 6, 'un': 7, 'tranquila': 8, 'samsa': 9, 'despertó': 10, 'tranquilo': 11, 'el': 12, 'no': 13, 'era': 14, 'normal': 15, 'y': 16, 'tarde': 17}

word_docs (diccionario de palabras y el número de documentos que aparecen.): defaultdict(<class 'int'>, {'una': 2, 'de': 2, 'gregor': 2, 'mañana': 2, 'samsa': 1, 'sueño': 2, 'tranquilo': 1, 'un': 2, 'despertó'

In [44]:
# 5. Transformación de los textos a secuencias de IDs enteros
# Aquí es donde el texto se convierte formalmente en vectores para alimentar la red neuronal.
secuencias_numéricas = tokenizer.texts_to_sequences(corpus)

# ==============================================================================
# REPORTE VISUAL COMPLETO EN LA CONSOLA
# ==============================================================================
print("=== DICCIONARIO INTERNO (WORD_INDEX) ===")
# Nota cómo Keras ordena los IDs por frecuencia (las que más aparecen tienen IDs más bajos)
for palabra, id_entero in diccionario_vocabulario.items():
    print(f"Palabra: '{palabra:<12}' ➡️ Asignada al ID: {id_entero}")

print("\n" + "="*70 + "\n")

print("=== RECONSTRUCCIÓN DE LOS DOCUMENTOS A SECUENCIAS ===")
for i, texto_original in enumerate(corpus):
    print(f"\nTexto original: \"{texto_original}\"")
    print(f"Secuencia IDs:  {secuencias_numéricas[i]}")

=== DICCIONARIO INTERNO (WORD_INDEX) ===
Palabra: '[UNK]       ' ➡️ Asignada al ID: 1
Palabra: 'una         ' ➡️ Asignada al ID: 2
Palabra: 'sueño       ' ➡️ Asignada al ID: 3
Palabra: 'gregor      ' ➡️ Asignada al ID: 4
Palabra: 'mañana      ' ➡️ Asignada al ID: 5
Palabra: 'de          ' ➡️ Asignada al ID: 6
Palabra: 'un          ' ➡️ Asignada al ID: 7
Palabra: 'tranquila   ' ➡️ Asignada al ID: 8
Palabra: 'samsa       ' ➡️ Asignada al ID: 9
Palabra: 'despertó    ' ➡️ Asignada al ID: 10
Palabra: 'tranquilo   ' ➡️ Asignada al ID: 11
Palabra: 'el          ' ➡️ Asignada al ID: 12
Palabra: 'no          ' ➡️ Asignada al ID: 13
Palabra: 'era         ' ➡️ Asignada al ID: 14
Palabra: 'normal      ' ➡️ Asignada al ID: 15
Palabra: 'y           ' ➡️ Asignada al ID: 16
Palabra: 'tarde       ' ➡️ Asignada al ID: 17


=== RECONSTRUCCIÓN DE LOS DOCUMENTOS A SECUENCIAS ===

Texto original: "Gregor Samsa despertó una mañana de un sueño tranquilo."
Secuencia IDs:  [4, 9, 10, 2, 5, 6, 7, 3, 11]

Texto or

La función `texts_to_matrix()` de `Tokenizer`:
- Crea un vector por documento provisto por entrada. 
- La longitud de los vectores es el tamaño total del vocabulario. 
- Proporciona el argumento `mode` que incluye:
- __`binary`__: si cada palabra está presente o no en el documento. Este es el valor predeterminado.
- __`count`__: el conteo de palabra en el documento.
- __`tfidf`__: la puntuación TF-IDF para cada palabra del documento.
- __`freq`__: La frecuencia de cada palabra.


In [56]:
# ==============================================================================
# 4. GENERACIÓN DE MATRICES Y DATAFRAMES POR MODO
# ==============================================================================

# --- MODO 1: COUNT (Conteo bruto de palabras) ---
matriz_count = tokenizer.texts_to_matrix(corpus, mode='count')
df_count = pd.DataFrame(matriz_count, columns=nombres_columnas)
df_count.index = [f"Documento_{i+1}" for i in range(len(corpus))]

# --- MODO 2: FREQ (Frecuencia relativa/porcentaje por renglón) ---
matriz_freq = tokenizer.texts_to_matrix(corpus, mode='freq')
df_freq = pd.DataFrame(matriz_freq, columns=nombres_columnas)
df_freq.index = [f"Documento_{i+1}" for i in range(len(corpus))]

# --- MODO 3: TFIDF (Ponderación estadística de Keras) ---
matriz_tfidf = tokenizer.texts_to_matrix(corpus, mode='tfidf')
df_tfidf = pd.DataFrame(matriz_tfidf, columns=nombres_columnas)
df_tfidf.index = [f"Documento_{i+1}" for i in range(len(corpus))]

# ==============================================================================
# 5. DESPLIEGUE DE REPORTES EN COLAB
# ==============================================================================
print("=== 1. MODO 'COUNT' (Frecuencias absolutas / Conteo bruto) ===")
display(df_count)

print("\n" + "="*80 + "\n")
print("=== 2. MODO 'FREQ' (Frecuencia relativa / Proporción del texto) ===")
# Muestra qué porcentaje del enunciado representa cada palabra
display(df_freq.round(4))

print("\n" + "="*80 + "\n")
print("=== 3. MODO 'TFIDF' (Pesos estadísticos calculados por Keras) ===")
# Recuerda que la fórmula interna de TF-IDF en Keras varía ligeramente de scikit-learn
display(df_tfidf.round(4))

=== 1. MODO 'COUNT' (Frecuencias absolutas / Conteo bruto) ===


,,[UNK],una,sueño,gregor,mañana,de,un,tranquila,samsa,despertó,tranquilo,el,no,era,normal,y,tarde
Documento_1,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
Documento_2,0.0,0.0,0.0,2.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0
Documento_3,0.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0




=== 2. MODO 'FREQ' (Frecuencia relativa / Proporción del texto) ===


,,[UNK],una,sueño,gregor,mañana,de,un,tranquila,samsa,despertó,tranquilo,el,no,era,normal,y,tarde
Documento_1,0.0,0.0,0.1111,0.1111,0.1111,0.1111,0.1111,0.1111,0.0000,0.1111,0.1111,0.1111,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Documento_2,0.0,0.0,0.0000,0.2222,0.1111,0.0000,0.1111,0.1111,0.0000,0.0000,0.0000,0.0000,0.1111,0.1111,0.1111,0.1111,0.0000,0.0000
Documento_3,0.0,0.0,0.2857,0.0000,0.0000,0.1429,0.0000,0.0000,0.2857,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1429,0.1429




=== 3. MODO 'TFIDF' (Pesos estadísticos calculados por Keras) ===


,,[UNK],una,sueño,gregor,mañana,de,un,tranquila,samsa,despertó,tranquilo,el,no,era,normal,y,tarde
Documento_1,0.0,0.0,0.6931,0.6931,0.6931,0.6931,0.6931,0.6931,0.0000,0.9163,0.9163,0.9163,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Documento_2,0.0,0.0,0.0000,1.1736,0.6931,0.0000,0.6931,0.6931,0.0000,0.0000,0.0000,0.0000,0.9163,0.9163,0.9163,0.9163,0.0000,0.0000
Documento_3,0.0,0.0,1.1736,0.0000,0.0000,0.6931,0.0000,0.0000,1.5514,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.9163,0.9163
